# Capstone — Which pages should be reviewed first for refresh?

Decision-support queue for Refresh / Content Opportunity Scoring, from May features to a June label at D = 2026-05-31.

 Abstract (5 sentences): I score labelable pages by how much attention they need so site maintainers know which ones to target first. I use May search + engagement aggregates from the FlyRank warehouse with the label `declined_30d_future` (June impressions under 80% of May). A depth-4 decision tree, grouped by client, ranks declining pages at P@100 0.98 vs the frozen Rule-2 baseline 0.89 and base rate 0.641. The win holds over 5 grouped seeds and survives a random-split audit; about 1 in 5 confident picks self-heal. The paper ends in a review-first queue with reason codes — momentum ranking, not proof a refresh will work.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Lane 2: Refresh / Content Opportunity Scoring

Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

I chose refresh/content opportunity scoring because it seems integral to FlyRank's core product. From what I've gathered, picking and ranking pages that a client should refresh in order to optimize traffic would need to use most of the observable data, and be critical to the business problem.

My lane *Refresh/Content Opportunity Scoring* is a combination of scoring and ranking. Fundamentally the task is to score the pages based on how much attention is needed, but also rank them based off these scores so site maintainers know which ones to target first.

The work empowers business' to choose which pages to first further optimize to improve overall SEO. Project managers/owners would mainly act on this recommendation; allocating resources appropriately to update the suggested pages. A wrong recommendation would mainly cost the time of the business. There is also the possibility that a refresh could not improve the SEO of a page, but have the inverse effect. Additionally a missed declining page would lead to continued traffic loss that could've been improved. There are many signals that impact the ranking of which pages need attention and these signals shift over time. This lends this problem to be hard to predict with a simple logic rule, and an effective problem for ML to tackle.

The pattern is too messy for an if-statement because there are many factors that go into which pages need to be refreshed. For example just observing the trend and ranking based on that might not tell the full story; an old outdated blogpost could have a worst trend than a real product page, but the product page may be more important to refresh first.

Success metric: Precision at 10 — how many of the top 10 pages the model ranks highest actually needed a refresh. See how many of the top 10 actually have a negative trend direction in the data. 

Careful words from the start: The work I will complete in the program will be creating a model able to observe trends based on the provided dataset and predict which pages are most in need of a refresh. This list prediction will help business owners and website builders allocate their SEO optimization efforts more efficiently; in a linear fashion. The work will not be able to predict Google/ChatGPT/Claude's recommendation engine or prove that if you refresh X page the SEO will improve by Y. It will only create predictions based on observable trends in FlyRank's dataset and be a good starting point for allocating SEO optimization efforts.

In [1]:
# Backs section 1: the decision, the action, the metric — no data needed.
print("lane: Refresh / Content Opportunity Scoring (score + rank)")
print("decision: which labelable pages to review first for refresh")
print("actor: project managers / owners allocating refresh work")
print("wrong call costs: wasted refresh time; missed decliner keeps losing traffic")
print("metric: precision@K (P@10 primary in W02; P@50/P@100 reported as the stable claim)")

lane: Refresh / Content Opportunity Scoring (score + rank)
decision: which labelable pages to review first for refresh
actor: project managers / owners allocating refresh work
wrong call costs: wasted refresh time; missed decliner keeps losing traffic
metric: precision@K (P@10 primary in W02; P@50/P@100 reported as the stable claim)


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

One row is one piece of content for one client's combined GSC and GA4 data on a given date. The dates are from 1/27/2025 until 6/30/2026.

Tables: `dim_clients` (104 clients) + `dim_content` + `fact_content_daily_performance` (78,835,655 rows) + `fact_content_query_90d`. I iterate on the two month partitions that cover the window (`month=2026-05` = prior, 23,381,448 May+Jun rows with June; `_sample` is June-only so it cannot feed the prior window on its own).

Decision point D = 2026-05-31. Features come from (D-30, D]; the label is June (D, D+30] so it was not knowable at decision time. Facts run through 2026-06-30, so the label window is fully observed.

What I excluded and why: sessions_paid — paid-for traffic should be irrelevant to refresh. ai_* vendor splits — AI traffic is measured but which vendor is irrelevant. IDs (`client_hash_id`, `content_hash_id`) group and split only, never features.

Data limits, in my words: All clients can't be compared equally because some clients have much more data than others; they start tracking at different times. 37.6% of rows have no available GA4 data and only GSC data. The zeroes in the GA4 columns thus aren't real tracked zeroes, they mean no data tracked. There's 6390 rows of duplicate data, so deduplication must be handled before any modelling.

Labelable filters (survivorship note, stated with every finding): prior_imp>=100, prior_obs>=7, future_obs>=7 — 100,785 of 333,275 window rows (30.3%, 41 of 52 clients). 232,490 low-volume/thin-history rows were filtered out, not scored as "not declined". Label: `declined_30d_future` = 1 when June impressions are under 80% of May; base rate 0.655 (0.641 on the grouped test side).

In [2]:
from pathlib import Path
import pandas as pd

def find_out():
    for c in [Path.cwd() / "work" / "outputs", Path.cwd().parent / "outputs", Path.cwd() / "outputs", Path("/Users/wyatt/Documents/programming/flyrank/work/outputs")]:
        if (c / "baseline_features.csv").exists():
            return c
    raise FileNotFoundError("baseline_features.csv missing")

OUT = find_out()
base = pd.read_csv(OUT / "baseline_features.csv", usecols=["labelable", "declined_30d_future", "client_hash_id", "content_hash_id"])
print(f"cache: baseline_features.csv | window rows {len(base):,} | labelable {int(base['labelable'].sum()):,} ({base['labelable'].mean():.1%}) | base rate {base['declined_30d_future'].mean():.3f}")
print(f"clients in window: {base['client_hash_id'].nunique()} | labelable clients: {base[base['labelable'] == True]['client_hash_id'].nunique()} (W04 observed 41)")
print("warehouse (observed W04): May+Jun partitions 23,381,448 rows | full fact 78,835,655 | 104 clients, 67 with GSC | 6,390 grain dups deduped")
assert base['content_hash_id'].is_unique, "one row per content expected"
print("check OK: one row per content; label window fully observed (D+30 == 2026-06-30).")

cache: baseline_features.csv | window rows 333,275 | labelable 100,785 (30.2%) | base rate 0.655
clients in window: 52 | labelable clients: 41 (W04 observed 41)
warehouse (observed W04): May+Jun partitions 23,381,448 rows | full fact 78,835,655 | 104 clients, 67 with GSC | 6,390 grain dups deduped
check OK: one row per content; label window fully observed (D+30 == 2026-06-30).


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

All features can be known on 2026-05-31. We generate prior-30d by aggregating this with static metadata. imp_ratio and pos_delta split the prior month in half (like the baseline). declined_30d_future consists of June and is only used for testing. We use one matrix for both models. We replace NaN with the median from the training set.

The two main models that fit the refresh/opportunity scoring track are decision tree and logistical regression, of which we will test both against the rule created in Week 4. Both models evaluate ranking at precision@k. A decision tree with depth 4 and min_samples_leaf 100 was the primary model. A standardize L2 logistical regression model was the comparison model. Both are fit on the same split and reported in one table against the baseline. The tree keeps its job only if it also wins the numbers; otherwise the table decides.

Baseline (frozen Rule 2, in plain words): a page is worth refreshing when its impressions are falling across the prior month **and** its position is slipping too — the classic fading star. Falling = second half under 80% of the first half's impressions; slipping = weighted position got worse. Reason codes: `IMPRESSIONS_FALLING`, `POSITION_SLIPPING`. Rule 2 stays a tie-break only, never a feature.

Validation: grouped by client (70/30, seed 42) so every row of a client stays on one side. The question becomes "does it work on a client it never saw?" — the deployable question. Week 6 holds everything fixed except the split: random rows (39 of 41 clients on BOTH sides, memorizes client character) vs grouped (0 shared). A pure time split is noted as future work.

Leakage hunt (three families, each ends in a number): label-derived columns (inject future/prior, watch the confession to 1.000, then remove it); future/overlapping windows (every feature tagged to end at D; query-table `impressions_90d`/`*_last30` CONTAIN June so only `*_prev30` would ever be safe — this work uses the daily fact only); product flags (none exist; Rule 2 breaks ties only). Solo-feature scan tops at imp_ratio 0.860 with ~0% ties — momentum, not a miracle.

In [3]:
# Backs section 3: the shipped feature set boundary (no future reaches the matrix).
NUM = ["prior_imp", "prior_ctr", "prior_days_with_impressions", "prior_position", "imp_ratio", "pos_delta", "prior_engaged_sessions", "prior_pageviews", "prior_ga4_obs_days", "word_count", "search_volume", "competition", "cpc", "content_age_days", "days_since_update", "has_keyword_data", "has_word_count"]
CAT = ["content_type", "main_intent"]
print(f"shipped matrix: {len(NUM)} numeric + 8 one-hot = 25 features | split: grouped by client, seed 42 | models: tree d=4 vs L2 logistic vs frozen Rule 2")
assert not {"future_imp", "future_obs_days", "declined_30d_future", "trend_direction", "trend_pct"} & set(NUM + CAT)
assert "client_hash_id" not in NUM + CAT and "content_hash_id" not in NUM + CAT
print("assert OK: no future_/declined_/trend_ column is a feature; IDs group only.")

shipped matrix: 17 numeric + 8 one-hot = 25 features | split: grouped by client, seed 42 | models: tree d=4 vs L2 logistic vs frozen Rule 2
assert OK: no future_/declined_/trend_ column is a feature; IDs group only.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Verdict: decision tree d=4 wins, beats baseline at every K, and beats logistic regression at P@50/P@100.

| model | P@10 | P@20 | P@50 | P@100 |
|---|---|---|---|---|
| baseline (Rule 2) | 0.80 | 0.85 | 0.86 | 0.89 |
| logistic regression | 0.80 | 0.85 | 0.84 | 0.79 |
| decision tree d=3 | 0.80 | 0.85 | 0.94 | 0.94 |
| decision tree d=4 | 1.00 | 1.00 | 0.98 | 0.98 |
| decision tree d=5 | 1.00 | 1.00 | 1.00 | 0.99 |
| base rate (grouped test, 12 clients) | 0.641 | | | |

Some observations from the data:

1. Logistic regression loses to the frozen rule at P@100 (0.79 vs 0.89).
2. The tree's perfect head is partly the tie-break. A depth-4 tree has only 16 distinct scores; the head of the ranking is the crash-leaf (imp_ratio ≤ 0.65) sorted by Rule 2's May impression loss. That leaf is 82.5% positive on test, not 100% — the perfect P@10/20 is the ranking on top of a strong-but-imperfect leaf.
3. The P@10 number flips with depth (0.80 at d=3, 1.00 at d=4/5). The stable claim is P@50/P@100 ≈ 0.94–0.99; the exact head is sensitive to the chosen tree size.

What the tree leans on: imp_ratio (the May mid-month collapse) is the tree's #1 split (importance 0.474) and logistic regression's #1 coefficient. Sanity check: a page that already lost most of its impressions in the second half of May usually keeps losing them in June — that is momentum, not magic, and it is not leakage (the ratio ends at D). content_age_days is the tree's second split (0.246): older pages are the classic refresh targets.

The audit keeps the verdict: the naive random split flatters logistic regression (+12 to +17 pts at P@50/P@100) and Rule 2 (+5 to +20 pts) because 39 of 41 clients sit on BOTH sides — client memorization. The tree is split-proof (gap ≈ 0 at P@50/P@100). Over 5 grouped seeds the tree holds P@50 0.972±0.016 and P@100 0.980±0.006 while Rule 2 spreads wider (P@100 0.924±0.025); the tree stayed above the baseline in every draw. Positive controls hit 1.000, so the harness does catch a leak when one exists.

In [4]:
# Backs section 4: the honest table, reprinted from the frozen W05/W06 runs (same slice, seed, tie-break).
print(f"{'model':<20}" + "".join(f"P@{k:>6}" for k in (10, 20, 50, 100)))
for name, vals in [("baseline (Rule 2)", (0.80, 0.85, 0.86, 0.89)), ("logistic reg", (0.80, 0.85, 0.84, 0.79)), ("tree d=4", (1.00, 1.00, 0.98, 0.98))]:
    print(f"{name:<20}" + "".join(f"{v:>7.3f}" for v in vals))
print(f"{'base rate (12 held-out clients)':<20}{0.641:>13.3f}")
print("5-seed grouped: tree P@50 0.972+-0.016, P@100 0.980+-0.006 | Rule2 P@100 0.924+-0.025 | before/after gap: tree ~0, logistic +17pts @100")

model               P@    10P@    20P@    50P@   100
baseline (Rule 2)     0.800  0.850  0.860  0.890
logistic reg          0.800  0.850  0.840  0.790
tree d=4              1.000  1.000  0.980  0.980
base rate (12 held-out clients)        0.641
5-seed grouped: tree P@50 0.972+-0.016, P@100 0.980+-0.006 | Rule2 P@100 0.924+-0.025 | before/after gap: tree ~0, logistic +17pts @100


## 5. Limitations

*What this work cannot claim.*

One snapshot, one window (May->June), ~12 held-out clients: directional evidence, not a guarantee. The head P@10-20 moves with seed and depth, so I only claim P@50/P@100.

The queue covers labelable pages only (100,785 of 333,275). Low-volume and thin-history pages were filtered out — this says nothing about them.

Refresh effect is NOT measured. Treated pages in history were CHOSEN, so any refresh-vs-not gap mixes choosing with treating. I did not run a refresh experiment and I do not predict Google's algorithm.

Where the model is most wrong: among test rows scored above 0.8 (22.6% of test), 19.1% did not actually decline — a May collapse that self-healed in June. A content team should treat the top of this list as "already bleeding," not "will bleed": about 1 in 5 of the confident picks recovers on its own.

In [5]:
import json
m = json.loads((OUT / "action_playbook_monitor.json").read_text())
print(f"labelable share: {m['n_labelable']:,} rows across {m['n_clients_labelable']} clients (30.2% of window) | recovery rate prob>0.8: {m['recovery_rate_prob_gt_08']:.1%} (~1-in-5)")
print("single window May->June; grouped test = 12 clients; head P@10-20 seed-sensitive ‒ claim P@50/P@100 only")
print("no causal claim: no refresh experiment; treated pages were CHOSEN (choosing-vs-treating bias)")

labelable share: 100,785 rows across 41 clients (30.2% of window) | recovery rate prob>0.8: 19.1% (~1-in-5)
single window May->June; grouped test = 12 clients; head P@10-20 seed-sensitive ‒ claim P@50/P@100 only
no causal claim: no refresh experiment; treated pages were CHOSEN (choosing-vs-treating bias)


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The queue suggests which pages to review first and does not guarantee refresh wins. What to do first? Tackle the pages at the top of the list going down.

BEFORE acting on any REVIEW_FIRST row, a human confirms: 1. Open the page: is it live, indexable, canonical — not redirected or mid-migration? 2. Check the dip: one bad week or a full-half collapse? Prefer WATCH when it looks like a single spike. 3. Check ownership and staleness: who owns it, was it edited elsewhere, is a campaign ending? 4. Check revenue/brand risk: top-traffic or sensitive pages need owner sign-off even at rank 1. 5. Log the decision so the next audit has a treatment record.

NO-GO list: never auto-publish; never act on non-labelable rows; never use query `*_last30` columns; never claim "refresh will recover X%"; never refresh solely for missing keyword/word-count; never compare raw ranks across clients as quality scores.

In [6]:
queue = pd.read_csv(OUT / "action_playbook_queue.csv")
print(f"review queue: {len(queue):,} rows (top 500) | all REVIEW_FIRST | 100% carry SHARP_DROP/FALLING")
print(queue.head(10)[["rank", "prob_decline", "imp_ratio", "pos_delta", "reason_codes"]].to_string(index=False))
print()
print(queue["reason_codes"].str.split("|").explode().value_counts().to_string())

review queue: 500 rows (top 500) | all REVIEW_FIRST | 100% carry SHARP_DROP/FALLING
 rank  prob_decline  imp_ratio  pos_delta                                                                 reason_codes
    1        0.9497      0.408       0.10                                 SHARP_DROP|POSITION_SLIPPING|THIN_ENGAGEMENT
    2        0.9497      0.137      -0.44                                                                   SHARP_DROP
    3        0.9497      0.222       4.57                                                 SHARP_DROP|POSITION_SLIPPING
    4        0.9497      0.183      -0.42                                       SHARP_DROP|STALE_90D|MISSING_WORDCOUNT
    5        0.9497      0.039       1.63 SHARP_DROP|POSITION_SLIPPING|OLD_PAGE_365D|THIN_ENGAGEMENT|MISSING_WORDCOUNT
    6        0.9497      0.275      25.20                                                 SHARP_DROP|POSITION_SLIPPING
    7        0.9497      0.056       1.62                                   SHARP_D

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper builds on three files in `work/outputs/`: the top-500 queue, the one honest claim, and the monitor baselines. The honest table (section 4) and the top-10 queue rows above are the two tables the page embeds.

In [8]:
import json, sys
sys.path.insert(0, str(OUT.parents[1] / "scripts"))
from ml_utils import simple_svg_bar_chart

# Honest table artifact (same numbers as section 4, with base rate).
honest = pd.DataFrame([
    {"model": "baseline (Rule 2)", "P@10": 0.80, "P@20": 0.85, "P@50": 0.86, "P@100": 0.89},
    {"model": "logistic reg", "P@10": 0.80, "P@20": 0.85, "P@50": 0.84, "P@100": 0.79},
    {"model": "decision tree d=4", "P@10": 1.00, "P@20": 1.00, "P@50": 0.98, "P@100": 0.98},
    {"model": "base rate (grouped test)", "P@10": 0.641, "P@20": 0.641, "P@50": 0.641, "P@100": 0.641},
])
honest.to_csv(OUT / "capstone_honest_table.csv", index=False)
simple_svg_bar_chart("P@100: tree vs baseline vs base rate (grouped held-out, n=12 clients)", ["tree d=4", "Rule 2", "logistic", "base rate"], [0.98, 0.89, 0.79, 0.641], OUT / "capstone_p100.svg")
for f in ["baseline_features.csv", "baseline_action_score.csv", "action_playbook_queue.csv", "action_playbook_summary.json", "action_playbook_monitor.json", "capstone_honest_table.csv", "capstone_p100.svg"]:
    print(("found " if (OUT / f).exists() else "MISSING ") + str(OUT / f))
print()
print("honest claim:", json.loads((OUT / "action_playbook_summary.json").read_text())["honest_claim"])

found /Users/wyatt/Documents/programming/flyrank/work/outputs/baseline_features.csv
found /Users/wyatt/Documents/programming/flyrank/work/outputs/baseline_action_score.csv
found /Users/wyatt/Documents/programming/flyrank/work/outputs/action_playbook_queue.csv
found /Users/wyatt/Documents/programming/flyrank/work/outputs/action_playbook_summary.json
found /Users/wyatt/Documents/programming/flyrank/work/outputs/action_playbook_monitor.json
found /Users/wyatt/Documents/programming/flyrank/work/outputs/capstone_honest_table.csv
found /Users/wyatt/Documents/programming/flyrank/work/outputs/capstone_p100.svg

honest claim: observed May->June on one snapshot: the model ranks/flags declining pages at P@100 ~0.98 on held-out clients; ~1-in-5 confident picks recover without action; head P@10-20 is seed-sensitive so the reliable claim is P@50/P@100; review-first candidates, not causal refresh effects


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.